# CHEM E 480 LAB 6

## NOTEBOOK OBJECTIVES

In this lab, you will:

- Implement and tune a PI controller on the TCLab system. 
- Demonstrate the need and implement an anti-reset windup strategy.

## EXERCISE 1: PI CONTROL

Proportional contorl (or P-control), which we looked at last week, looks at the error, which is the difference between the temperature set point and the current value (where error is calculated as $e(t) = T_{sp} - T_m$), and sends a controller signal proportional to that error in order to change the manipulated variable via the gain, $K_c$. Integral control takes into account the history of the error signal over time by adding a term that integrates the error, and the tunable parameter $\tau_I$ i the integral time, or reset time, that controls how long this error memory is held onto.

Design and implement a  PI controller (without anti-reset windup) on the Q1/T1 system. PI control uses the following equation to determine the heater output ($Q$) over time:

$$Q(t) = Q_{ss} + K_c \left( e(t) + \frac{1}{\tau_I} \int^t_0 e(t^*)dt^* \right)$$

where $Q$ is the controller output (heater value), $Q_{ss}$ is the controller output (heater value) at steady state initially, $K_c$ is the controller gain, and $\tau_I$ is the controler time constant for the integral control.

You may model this as:

$$Q(t) = Q_{ss} + K_Pe_{current} + K_I \left( e_{cumulative} + e_{current}\Delta t \right)$$

where 

$$K_P = K_c$$

$$K_I = \frac{K_c}{\tau_I}$$

Try different controller settings (using various values of $K_c$ and $\tau_I$). Demonstrate the need for anti-reset windup by putting in a large disturbance (from Q2) or a large change in setpoint which saturates the controller for a long time.

### IMPORTS

In [ ]:
import tclab
import numpy as np
import time
import matplotlib.pyplot as plt
from scipy.integrate import odeint

### CREATE FUNCTIONS

First create a function to run the PI controller.

In [ ]:
def pi(sp, pv, ierr, dt, params):
    '''
    The goal of this function is to calulate the output of a PI controller.
    
    INPUTS
    
        sp (float) - temperature setpoint
        pv (float) - current temperature
        pv_last (float) - prior temperature
        ierr (float) - integral error
        dt (float) - time increment between measurements
        params (array) - list of parameters that give [Kc, tauI]
    
    OUTPUTS
    
        op (float) - output of the PID controller
        P (float) - proportional contribution
        I (float) - integral contribution
    '''
    # Unpack parameters from params array
    Kc = # UNPACK FROM params ARRAY
    tauI = # UNPACK FROM params ARRAY
    
    # Parameters in terms of PID coefficients
    KP = # DEFINE WITH EQUATIONS ABOVE USING PARAMS FROM ARRAY
    KI = # DEFINE WITH EQUATIONS ABOVE USING PARAMS FROM ARRAY
    
    # ubias for controller (initial heater)
    op0 = # SET INITIAL HEATER VALUE
    
    # Set upper and lower bounds on heater level
    ophi = # SET MAX HEATER VALUE
    oplo = # SET MIN HEATER VALUE
    
    # Calculate the error
    error = # CALCULATE ERROR AS DIFFERENCE BETWEEN SET POINT AND CURRENT TEMPERATURE
    
    # Calculate the integral error
    ierr = # ADD NEW CALCULATED ERROR (equation above) TO CUMULATIVE ERROR (ierr) TO GET NEW CUMULATIVE ERROR (ierr)
    
    # Calculate the PI output
    P = # OUTPUT FROM JUST P CONTROL
    I = # OUTPUT FROM JUST I CONTROL
    op = # SUMMED HEATER OUTPUT (ABOVE EQUATION)
    
    # Implement anti-reset windup (this is done for you)
    # Run once without this (keep in comment block) and once without (remove from comment block)
    '''
    if op < oplo or op > ophi:
        I = I - KI * error * dt
        
        # Clip output
        op = max(oplo,min(ophi,op))
    '''
        
    # Return the controller output and PI terms
    return [op, P, I]

I've gone ahead and created a function for you to save the data to a text file.

In [ ]:
def save_txt(t, u1, u2, y1, y2, sp1, sp2):
    '''
    The goal of this function is to save the data and set point to a text file.
    
    INPUTS
        
        t - time
        u1 - heater 1 level
        u2 - heater 2 level
        y1 - T1 output
        y2 - T2 output
        sp1 - set-point for T1
        sp2 - set-point for T2
    
    OUTPUTS
        
        None
    '''
    
    data = np.vstack((t, u1, u2, y1, y2, sp1, sp2))  # vertical stack
    data = data.T  # transpose data
    top = ('Time,Q1,Q2,T1,T2,TSP1,TSP2')
    np.savetxt('validate.txt', data, delimiter=',',header=top, comments='')
    
    return

### SET VALUES FOR PI PARAMETERS

In [ ]:
# PI Parameters
Kc = # SET A VALUE
tauI = # SET A VALUE # sec

### RUN ARDUINO

In [ ]:
# Connect to Arduino
lab = tclab.TCLab()

# Turn LED on
print('LED On')
lab.LED(100)

# Set run time in minutes
run_time = # RUN FOR 10 MINUTES

# Set number of cycles
loops = int(60.0*run_time)
tm = np.zeros(loops)

# Set temperature set point (degC)
Tsp1 = np.ones(loops) * # SENSOR MEASUREMENT FOR T1

# Set temperature set point steps
Tsp1[# AFTER 3 SECONDS AND ONWARD] = # SET TO 50
Tsp1[# AFTER 300 SECONDS AND ONWARD] = # SET TO 40

T1 = np.ones(loops) * # SENSOR MEASUREMENT FOR T1 # measured T (degC)
error_sp = np.zeros(loops)

Tsp2 = np.ones(loops) * # SENSOR MEASUREMENT FOR T2 # set point (degC)
T2 = np.ones(loops) * # SENSOR MEASUREMENT FOR T2 # measured T (degC)

# Impulse tests (0 - 100%)
Q1 = np.ones(loops) * 0.0
Q2 = np.ones(loops) * 0.0

print('Running Main Loop. Ctrl-C to end.')
print('  Time     SP     PV     Q1   =  P   +  I  +   D    IAE')
print(('{:6.1f} {:6.2f} {:6.2f} ' + \
       '{:6.2f} {:6.2f} {:6.2f} {:6.2f} {:6.2f}').format( \
           tm[0], Tsp1[0], T1[0], \
           Q1[0], 0.0, 0.0,0.0, 0.0))

# Main Loop
start_time = time.time()
prev_time = start_time
dt_error = 0.0

# Set initial integral error
ierr = # INITIAL ERROR VALUE BEFORE STARTING

# Set initial integral absolute error
iae = # INITIAL ABSOLUTE ERROR VALUE BEFORE STARTING

# Create plot
plt.figure(figsize=(10,7))
plt.ion()
plt.show()

try:
    
    for i in range(1,loops):
        
        # Sleep time
        sleep_max = 1.0
        sleep = sleep_max - (time.time() - prev_time) - dt_error
        
        if sleep >= 1e-4:
            time.sleep(sleep - 1e-4)
            
        else:
            print('exceeded max cycle time by ' + str(abs(sleep)) + ' sec')
            time.sleep(1e-4)

        # Record time and change in time
        t = time.time()
        dt = # CALCULATE DELTA TIME AS CURRENT TIME MINUS PREVIOUS TIME
        
        if (sleep >= 1e-4):
            dt_error = dt - 1.0 + 0.009
            
        else:
            dt_error = 0.0
            
        prev_time = # SET VALUE OF PREVIOUS TIME TO BE CURRENT TIME
        tm[i] = # CALCULATE TOTAL TIME AS CURRENT TIME MINUS START TIME

        # Read temperatures in Celcius
        T1[i] = # LAB SENSOR FOR T1
        T2[i] = # LAB SENSOR FOR T2

        # Calculate integral absolute error
        iae += np.abs(# CALCULATE ERROR BETWEEN CURRENT SET POINT FOR T1 VALUE AND CURRENT T1 VALUE)

        # Calculate PI output
        [Q1[i], P, ierr] = pi(# SET POINT T1, # T1 VALUE, # ERROR, # DELTA TIME, # PARAMS ARRAY)

        # Write output (0-100)
        lab.Q1(# SET NEW VALUE OF HEATER)
        lab.Q2(# KEEP HEATER 2 OFF)

        # Print line of data
        print(('{:6.1f} {:6.2f} {:6.2f} ' + \
              '{:6.2f} {:6.2f} {:6.2f} {:6.2f}').format( \
                  tm[i], Tsp1[i], T1[i], \
                  Q1[i], P, ierr, iae))

        # Update plot
        plt.clf()
        
        # Plot T1 set point and measured value
        ax=plt.subplot(2,1,1)
        ax.grid()
        plt.plot(# TIME TO CURRENT LOOP POINT, # T1 SET POINT TO CURRENT LOO POINT, 'k--', label=r'$T_1$ set point')
        plt.plot(# TIME TO CURRENT LOOP POINT, # T1 VALUE TO CURRENT LOOP POINT, 'r.', label=r'$T_1$ measured')
        plt.ylabel(r'Temperature ($^oC$)')
        plt.legend(loc=4)
        
        # Plot heater 1 set value over time
        ax=plt.subplot(2,1,2)
        ax.grid()
        plt.plot(# TIME TO CURRENT LOOP POINT, # HEATER VALUE TO CURRENT LOOP POINT, 'b-', label=r'$Q_1$')
        plt.ylabel('Heater (%)')
        plt.legend(loc=1)
        plt.xlabel('Time (sec)')
        plt.draw()
        plt.pause(0.05)

    # Turn off heaters
    lab.Q1(0)
    lab.Q2(0)
    lab.close()
    
    # Save text file
    save_txt(tm[0:i], Q1[0:i], Q2[0:i], T1[0:i], T2[0:i], Tsp1[0:i], Tsp2[0:i])
    
    # Save figure
    plt.savefig('PI_Control_Q1T1.png')

# Allow user to end loop with Ctrl-C
except KeyboardInterrupt:
    
    # Disconnect from Arduino
    lab.Q1(0)
    lab.Q2(0)
    
    # Print shut down message
    print('Shutting down')
    lab.close()
    
    # Save data
    save_txt(tm[0:i], Q1[0:i], Q2[0:i], T1[0:i], T2[0:i], Tsp1[0:i], Tsp2[0:i])
    
    # Save figure
    plt.savefig('PI_Control_Q1T1.png')

# Make sure serial connection closes with an error
except:
    
    # Disconnect from Arduino
    lab.Q1(0)
    lab.Q2(0)
    
    # Print error message
    print('Error: Shutting down')
    lab.close()
    
    # Save data
    save_txt(tm[0:i], Q1[0:i], Q2[0:i], T1[0:i], T2[0:i], Tsp1[0:i], Tsp2[0:i])
    
    # Save figure
    plt.savefig('PI_Control_Q1T1.png')
    
    raise

## EXERCISE 2: PI CONTROL CONTINUED

Implement and tune a PI controller on the Q1/T2 system. Experiment with its performance and tuning of the controller. This will use the same `pi` function from above, but instead you'll be turning on heater 1 to control temperature sensor 2.

### SET VALUES FOR PI PARAMETERS

In [ ]:
# PI Parameters
Kc = # SET A VALUE
tauI = # SET A VALUE # sec

### RUN ARDUINO

In [ ]:
# Connect to Arduino
lab = tclab.TCLab()

# Turn LED on
print('LED On')
lab.LED(100)

# Set run time in minutes
run_time = # 10 MINUTES

# Set number of cycles
loops = int(60.0*run_time)
tm = np.zeros(loops)

# Set temperature set point (degC)
Tsp2 = np.ones(loops) * # SENSOR FOR T2

# Set temperature set point steps
Tsp2[# AFTER 3 SECONDS AND ONWARD] = # SET TO 50
Tsp2[# AFTER 300 SECONDS AND ONWARD] = # SET TO 40

Tsp1 = np.ones(loops) * # SENSOR FOR T1 # set point (degC)
T1 = np.ones(loops) * # SENSOR FOR T1 # measured T (degC)

T2 = np.ones(loops) * # SENSOR FOR T2 # measured T (degC)
error_sp = np.zeros(loops)

# Impulse tests (0 - 100%)
Q1 = np.ones(loops) * 0.0
Q2 = np.ones(loops) * 0.0

print('Running Main Loop. Ctrl-C to end.')
print('  Time     SP     PV     Q1   =  P   +  I  +   D    IAE')
print(('{:6.1f} {:6.2f} {:6.2f} ' + \
       '{:6.2f} {:6.2f} {:6.2f} {:6.2f} {:6.2f}').format( \
           tm[0], Tsp2[0], T2[0], \
           Q1[0], 0.0, 0.0, 0.0, 0.0))

# Main Loop
start_time = time.time()
prev_time = start_time
dt_error = 0.0

# Set initial integral error
ierr = # INITIAL ERROR VALUE BEFORE STARTING

# Set initial integral absolute error
iae = # INITIAL ABSOLUTE ERROR VALUE BEFORE STARTING

# Create plot
plt.figure(figsize=(10,7))
plt.ion()
plt.show()

try:
    
    for i in range(1,loops):
        
        # Sleep time
        sleep_max = 1.0
        sleep = sleep_max - (time.time() - prev_time) - dt_error
        
        if sleep >= 1e-4:
            time.sleep(sleep - 1e-4)
            
        else:
            print('exceeded max cycle time by ' + str(abs(sleep)) + ' sec')
            time.sleep(1e-4)

        # Record time and change in time
        t = time.time()
        dt = # CALCULATE DELTA TIME AS CURRENT TIME MINUS PREVIOUS TIME
        
        if (sleep >= 1e-4):
            dt_error = dt - 1.0 + 0.009
            
        else:
            dt_error = 0.0
            
        prev_time = # SET VALUE OF PREVIOUS TIME TO BE CURRENT TIME
        tm[i] = # CALCULATE TOTAL TIME AS CURRENT TIME MINUS START TIME

        # Read temperatures in Celcius
        T1[i] = # SENSOR FOR T1
        T2[i] = # SENSOR FOR T2
    
        # Calculate integral absolute error
        iae += np.abs(# CALCULATE ERROR BETWEEN CURRENT SET POINT FOR T2 VALUE AND CURRENT T2 VALUE)

        # Calculate PI output
        [Q1[i], P, ierr] = pi(# SET POINT T2, # T2 VALUE, # ERROR, # DELTA TIME, # PARAMS ARRAY)

        # Write output (0-100)
        lab.Q1(# SET NEW VALUE OF HEATER)
        lab.Q2(# KEEP HEATER 2 OFF)

        # Print line of data
        print(('{:6.1f} {:6.2f} {:6.2f} ' + \
              '{:6.2f} {:6.2f} {:6.2f} {:6.2f}').format( \
                  tm[i], Tsp2[i], T2[i], \
                  Q1[i], P, ierr, iae))

        # Update plot
        plt.clf()
        
        # Plot T2 set point and measured value
        ax=plt.subplot(2,1,1)
        ax.grid()
        plt.plot(# TIME TO CURRENT LOOP POINT, # T2 SET POINT TO CURRENT LOOP POINT, 'k--', label=r'$T_2$ set point')
        plt.plot(# TIME TO CURRENT LOOP POINT, # T2 MEASURED VALUE TO CURRENT LOOP POINT, 'r.', label=r'$T_2$ measured')
        plt.ylabel(r'Temperature ($^oC$)')
        plt.legend(loc=4)
        
        # Plot heater 1 set value over time
        ax=plt.subplot(2,1,2)
        ax.grid()
        plt.plot(# TIME TO CURRENT LOOP POINT, # HEATER 1 VALUE TO CURRENT LOOP POINT,'b-',label=r'$Q_1$')
        plt.ylabel('Heater (%)')
        plt.legend(loc=1)
        plt.xlabel('Time (sec)')
        plt.draw()
        plt.pause(0.05)

    # Turn off heaters
    lab.Q1(0)
    lab.Q2(0)
    lab.close()
    
    # Save text file
    save_txt(tm[0:i], Q1[0:i], Q2[0:i], T1[0:i], T2[0:i], Tsp1[0:i], Tsp2[0:i])
    
    # Save figure
    plt.savefig('PI_Control_Q1T2.png')

# Allow user to end loop with Ctrl-C
except KeyboardInterrupt:
    
    # Disconnect from Arduino
    lab.Q1(0)
    lab.Q2(0)
    
    # Print shut down message
    print('Shutting down')
    lab.close()
    
    # Save data
    save_txt(tm[0:i], Q1[0:i], Q2[0:i], T1[0:i], T2[0:i], Tsp1[0:i], Tsp2[0:i])
    
    # Save figure
    plt.savefig('PI_Control_Q1T2.png')

# Make sure serial connection closes with an error
except:
    
    # Disconnect from Arduino
    lab.Q1(0)
    lab.Q2(0)
    
    # Print error message
    print('Error: Shutting down')
    lab.close()
    
    # Save data
    save_txt(tm[0:i], Q1[0:i], Q2[0:i], T1[0:i], T2[0:i], Tsp1[0:i], Tsp2[0:i])
    
    # Save figure
    plt.savefig('PI_Control_Q1T2.png')
    
    raise